<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.2A/MP8.2A_04_report.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.2A · Thermal Cycling with Natural Convection
## 04 · The Report

*Mini project of the course **Deep Learning for Engineering**, Part 2. Five notebooks, run in order:*

00 The Problem → 01 The Data and the Physics → 02 The Reference Model and the Ground Truth → 03 The Physics-Informed Network → **04 The Report**

*Notebooks 00 to 02 are given and solved: read them, run them, and use what they build. Notebooks 03 and 04 are yours.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp82a.py', 'MP8.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp82a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp82a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.keep_outputs("MP8.2A_outputs")                            # results kept between the notebooks (Google Drive on Colab)
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

## 1 · What this notebook does

It reads three files - your network's answers from notebook 03 (`MP8.2A_results.npz`), the reference model's time from notebook 02 (`MP8.2A_reference.json`), and the ground truth (`MP8.2A_truth.npz`) - and asks you to turn them into the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# The three files
import json
missing = cc.needed("MP8.2A_results.npz", "MP8.2A_reference.json")
assert not missing, f"run notebooks 02 and 03 first: missing {missing}"
R = np.load(cc.output_path("MP8.2A_results.npz"))
REF = json.load(open(cc.output_path("MP8.2A_reference.json")))
G = np.load("MP8.2A_truth.npz")
print("in R:", ", ".join(R.files))
print(f"reference model (notebook 02): worst {REF['worst_ref_K']:.2f} K, {REF['t_ref_s']*1e3:.1f} ms per new power")

## 2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the worst peak error and the worst trough error at each test power (K);
* the largest error of the field at 600 s at 165 W (K);
* the network's online time for one new power, the reference model's on the same runtime (both in `R`), and their ratio;
* the offline training time.

Then say in one line each whether the three requirements of notebook 00 are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.2f}")

## 3 · TODO 5 · The report's figures

Draw, and save as `MP8.2A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the centre's history against the ground truth at the three test powers, with the error; the error of the field at 600 s; the online times of the network and the reference model for 1, 10 and 100 powers (`R["net_1"]`, `R["ref_10"]` and so on); and the swing of the tenth cycle against the power (`R["centre_sweep"]`, question 16).

In [ ]:
# TODO 5 - the report's figures
...

## The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 10 are about your network and its result; 11 to 17 ask you to use it; 18 concludes. The next cell writes `MP8.2A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features you computed from them), layers, neurons and number of weights. Why this size?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. How did you handle the twenty switches of the duty cycle? Give the worst peak error with your remedy and without it.": """
""",
    "4. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time on the T4 in seconds.": """
""",
    "5. How did you choose the weights of the loss terms? Show the terms during training in one figure and say which was hardest to bring down.": """
""",
    "6. The peaks and troughs at 115, 165 and 195 W: give the worst error at each power in kelvin. Are all within 0.3 K?": """
""",
    "7. In which cycle is your error largest, and why there?": """
""",
    "8. The field at 600 s at 165 W: where is your largest error, how large, and why there?": """
""",
    "9. The online time to answer a new power: your network against the reference model on the same runtime, in milliseconds, and the ratio. What does each spend its time on?": """
""",
    "10. Ask for 1, 10 and 100 new powers at once: how do the network's and the reference model's times grow? Give the three times of each, and say why they grow so differently.": """
""",
    "11. How much does natural convection lower the tenth peak at 165 W, against the fan alone (S = 0, Ex08's model)? Give both peaks in kelvin.": """
""",
    "12. Your network's energy balance over the window at 165 W: heat in, through the fins, stored, in joules. How far from closing?": """
""",
    "13. Outside the training range: your errors at 90 W and 210 W against the ground truth. How fast does the error grow outside the range?": """
""",
    "14. Single against double precision: train or evaluate in float32. What happens to the accuracy and to the online time?": """
""",
    "15. Train again with another seed. How much do the worst errors of the two runs differ? Is your result reproducible within 0.3 K?": """
""",
    "16. The peak, the trough and the swing of the tenth cycle against the power from 100 to 200 W, from your network. Fit each as a P^n: what are the three n, and what do they say about how strongly natural convection changes the answer across this range of powers?": """
""",
    "17. If the module were on for 30 s instead of 20 s, could your network answer? What would you change to make it answer any duty cycle?": """
""",
    "18. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP8.2A — Thermal Cycling with Natural Convection", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP8.2A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP8.2A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP8.2A_report.md into MP8.2A_report.pdf, with any figure
# saved as MP8.2A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP8.2A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP8.2A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP8.2A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP8.2A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass